In [0]:
%run ./01_setup_paths_and_schema

In [0]:


from pyspark.sql.functions import *

df = spark.read.format("delta") \
.load(f"{bronze_path}/claims")

expected = StructType(

claims_schema.fields +

[

StructField(
"ingestion_time",
TimestampType(),
True
)

]

)

missing,datatype,extra = validate_schema(

df,

expected

)

print(missing)

print(datatype)

print(extra)


bad = df.filter(

col("claim_id").isNull()

|

col("patient_id").isNull()

|

col("claim_amount").isNull()

|

(col("claim_amount") <=0)

|

(~col("claim_status").isin(

"Approved",

"Pending",

"Rejected"

))

|

col("claim_date").isNull()

)

bad.write \
.format("delta") \
.mode("overwrite") \
.save(

f"{quarantine_path}/claims"

)

good = df.subtract(

bad

)

good.write \
.format("delta") \
.mode("overwrite") \
.save(

f"{validated_path}/claims"

)